In [1]:
!pip install retina-face librosa opencv-python ffmpeg-python tqdm scikit-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
import ffmpeg
import zipfile
from tqdm import tqdm
from retinaface import RetinaFace

In [25]:
ZIP_PATH = "/content/drive/MyDrive/Deepfake/FakeAVCeleb.zip"

EXTRACT_PATH = "/content/FakeAVCeleb_v1.2"

OUTPUT_PATH = "/content/drive/MyDrive/Deepfake/processed_dataset_v7"

os.makedirs(OUTPUT_PATH, exist_ok=True)

FRAME_SIZE = 224
NUM_FRAMES = 16
NUM_AUDIO_SEG = 16
SR = 16000

In [5]:
if not os.path.exists(EXTRACT_PATH):

    print("Extracting dataset...")

    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content")

print("Dataset ready")

Extracting dataset...
Dataset ready


In [26]:
def detect_face_box(frame):

    try:

        faces = RetinaFace.detect_faces(frame)

        if isinstance(faces, dict):

            face = list(faces.values())[0]

            return face["facial_area"]

    except:
        pass

    return None

In [27]:
def extract_frames(video):

    cap = cv2.VideoCapture(video)

    frames=[]
    face_box=None
    frame_count=0

    while True:

        ret,frame = cap.read()

        if not ret:
            break

        frame_count += 1

        if face_box is None and frame_count <= 30:
            face_box = detect_face_box(frame)

        if face_box is not None:

            x1,y1,x2,y2 = face_box
            h,w,_ = frame.shape

            x1=max(0,x1)
            y1=max(0,y1)
            x2=min(w,x2)
            y2=min(h,y2)

            crop = frame[y1:y2,x1:x2]

        else:

            h,w,_ = frame.shape
            size=min(h,w)//2
            cx=w//2
            cy=h//2

            crop = frame[
                cy-size:cy+size,
                cx-size:cx+size
            ]

        crop = cv2.resize(crop,(FRAME_SIZE,FRAME_SIZE))

        frames.append(crop)

        if len(frames) >= NUM_FRAMES:
            break

    cap.release()

    return frames

In [28]:
def extract_audio(video,tmp):

    (
        ffmpeg
        .input(video)
        .output(tmp,ac=1,ar=SR)
        .overwrite_output()
        .run(quiet=True)
    )

In [29]:
def process_audio(path):

    audio, sr = librosa.load(path, sr=SR)

    if len(audio) < NUM_AUDIO_SEG:
        return []

    seg_len = len(audio) // NUM_AUDIO_SEG

    feats = []

    TARGET_T = 32

    for i in range(NUM_AUDIO_SEG):

        s = i * seg_len
        e = s + seg_len

        segment = audio[s:e]

        mel = librosa.feature.melspectrogram(
            y=segment,
            sr=sr,
            n_mels=128
        )

        mel = librosa.power_to_db(mel)

        if mel.shape[1] < TARGET_T:

            pad = TARGET_T - mel.shape[1]

            mel = np.pad(
                mel,
                ((0,0),(0,pad)),
                mode="constant"
            )

        else:

            mel = mel[:, :TARGET_T]

        feats.append(mel)

    return feats

In [30]:
videos=[]

for root,dirs,files in os.walk(EXTRACT_PATH):

    for f in files:

        if f.endswith(".mp4"):

            videos.append((os.path.join(root,f),root))

print("Videos found:",len(videos))

Videos found: 21544


In [31]:
def get_label(folder):

    folder=folder.lower()

    if "realvideo-realaudio" in folder:
        return 0,0,0

    if "fakevideo-realaudio" in folder:
        return 1,0,1

    if "realvideo-fakeaudio" in folder:
        return 0,1,2

    if "fakevideo-fakeaudio" in folder:
        return 1,1,3

    return None

In [32]:
processed=set(
    f.replace(".npz","")
    for f in os.listdir(OUTPUT_PATH)
    if f.endswith(".npz")
)

In [ ]:
metadata=[]

for video,folder in tqdm(videos):

    vid=os.path.basename(video).split(".")[0]
    folder_name=os.path.basename(folder)

    unique_id=f"{folder_name}_{vid}"

    if unique_id in processed:
        continue

    try:

        frames=extract_frames(video)

        if len(frames) < 8:
            continue

        tmp=f"/content/{unique_id}.wav"

        extract_audio(video,tmp)

        audio=process_audio(tmp)

        if len(audio)==0:
            continue

        lbl=get_label(folder)

        if lbl is None:
            continue

        vfake,afake,cls=lbl

        path=os.path.join(OUTPUT_PATH,f"{unique_id}.npz")

        np.savez_compressed(
            path,
            frames=np.array(frames),
            audio=np.array(audio),
            label=cls
        )

        metadata.append({
            "video_id":unique_id,
            "video_fake":vfake,
            "audio_fake":afake,
            "class_label":cls,
            "frames":len(frames)
        })

    except Exception as e:

        print("Error:",video)
        continue

100%|██████████| 21544/21544 [2:08:14<00:00,  2.80it/s]


In [ ]:
files=[
    f.replace(".npz","")
    for f in os.listdir(OUTPUT_PATH)
    if f.endswith(".npz")
]

df=pd.DataFrame({"video_id":files})

df.to_csv(
    os.path.join(OUTPUT_PATH,"metadata.csv"),
    index=False
)

print("Total samples:",len(df))

Total samples: 20558


In [ ]:
sample_file=os.listdir(OUTPUT_PATH)[0]

sample=np.load(
    os.path.join(OUTPUT_PATH,sample_file)
)

print(sample["frames"].shape)
print(sample["audio"].shape)
print(sample["label"])

(16, 224, 224, 3)
(16, 128, 32)
1


In [ ]:
print(
len([f for f in os.listdir(OUTPUT_PATH) if f.endswith(".npz")])
)

20558
